# 00 - 项目概述：多模态内容安全审核系统

**目标**：构建面向 TikTok 审核场景的多模态内容安全审核系统——从数据构建到模型训练到量化评估的完整闭环。

本 Notebook 讲解 7 个核心概念，帮助理解项目的技术基础和设计决策。

---
## 概念 1：本项目在 AI 训练链路中的位置

大模型的训练分为两大阶段：

```
阶段 1: Pre-train 数据清洗 → 海量无标签网页去脏（处理 TB 级数据）
阶段 2: Pre-train 训练 → 产出 Base Model（只会接龙，不会对话）
阶段 3: Post-train 数据准备 ← 本项目在这里
  ├── 构建 SFT 指令数据（教模型对话格式）
  ├── 构建安全数据（教模型什么该拒绝）← 核心
  └── 构建偏好对数据（教模型什么是好回复）
阶段 4: Post-train 训练 → SFT → DPO → Chat Model
```

**类比**：Pre-train 数据清洗是"洗菜"（去掉网页中的垃圾内容），本项目是"配调料"（精心标注什么是有害的、什么是安全的）。

**关键区别**：
- Pre-train 数据清洗处理 **无标签** 数据，追求去掉低质量
- 本项目处理 **有标签** 数据，追求安全分类的准确性和完整性
- 数据量差异巨大：Pre-train 处理 TB 级，本项目处理 GB 级但标注精细

---
## 概念 2：多模态安全的核心挑战——模态缝隙

**核心发现（MM-SafetyBench, ECCV 2024）**：单独看文本无害、单独看图片无害，但组合在一起有害。

### 三种多模态攻击形态

| 攻击类型 | 描述 | TikTok 场景举例 |
|---------|------|----------------|
| **Vanilla（直接文本）** | 直接用有害文本提问 | 评论区直接发布有害内容 |
| **Typographic/FigStep** | 将有害文本渲染成图片 | 用字幕卡片展示有害指令 |
| **Query-Relevant Image** | 语义相关图片配有害文本 | 暴力场景视频配煽动性文字 |

### 为什么安全对齐不能自动迁移到视觉模态？

```
纯文本 LLM："How to make a bomb?" → 拒绝 ✓
多模态 LLM：[图片上写着 'bomb 制作步骤'] + "请按照图片操作" → 回答了！✗
```

**原因**：安全约束只在文本端训练，没有被传递到视觉编码器端。LLM 处理图片中的文字时，绕过了文本安全检查。

**数据**：MM-SafetyBench 发现安全对齐的 LLM 接上视觉模块后，攻击成功率仍达 **40-70%**。

---
## 概念 3：TikTok 审核的多模型级联架构

TikTok 每天处理数百万条内容上传，不可能每条都用大模型审核。工业界采用**级联架构**：

```
用户上传内容
  ↓
第一层：轻量级分类器（DistilBERT 级别，~66M 参数，毫秒级）
  ← 本项目训练的就是这一层
  极快，初筛 90% 的明显安全/明显有害内容
  ↓ （剩余 10% 不确定的）
第二层：中等模型（7B 级别，百毫秒级）
  处理第一层不确定的内容，做更细的分类
  ↓ （剩余 1-2% 边界案例）
第三层：大模型 / 人工复核（秒级）
  处理最难判断的边界案例
  ↓
反馈回路：人工复核结果 → 更新训练数据 → 重新训练第一层
```

### 为什么第一层用 DistilBERT 而不是 GPT-4？

| 指标 | DistilBERT | GPT-4 |
|------|-----------|-------|
| 参数量 | ~66M | ~1.8T |
| 推理延迟 | <10ms | >1s |
| 每百万条成本 | ~$1 | ~$10,000 |
| 准确率 | ~85-90% | ~95%+ |

**结论**：用轻量模型处理大多数明显案例，用重量模型处理困难案例，是成本和效果的最优平衡。

---
## 概念 4：对比构造——为什么不能只有有害数据

### Over-refusal 问题

如果训练数据只有有害样本，模型会学到"凡是涉及敏感词的都拒绝"，导致 **over-refusal**（过度拒绝）：

```
有害："How to make a bomb at home?" → 应该拒绝 ✓
无害："How to make a bath bomb at home?" → 也被拒绝了 ✗
无害："How do I kill a Python process?" → 也被拒绝了 ✗
无害："What's the best way to destroy old hard drives?" → 也被拒绝了 ✗
```

### WildJailbreak 的对比构造方法

为每一条有害数据配一条"表面相似但实际无害"的对比数据：

| 有害 Prompt | 对比无害 Prompt |
|------------|----------------|
| How to make a bomb? | How to make a bath bomb? |
| How to kill someone | How to kill a process |
| How to stalk a person | How to stalk prey as a wildlife photographer |

### TikTok 审核中 Over-refusal 的代价

如果审核系统 over-refuse，会误删大量正常内容：
- 美食视频"如何做爆炸鸡排"被误删
- 科普视频"病毒如何传播"被误删
- 游戏视频"如何击杀 Boss"被误删

→ 用户体验下降 → 创作者流失 → 平台损失

---
## 概念 5：OCR + 多模态双层防御

对于图片中包含文字的安全威胁，有两种检测方案：

### 方案 A：OCR（光学字符识别）
```
图片 → OCR 提取文字 → 文本安全分类器 → 判定
```
- **优点**：便宜、快速、技术成熟
- **缺点**：手写体、艺术字、emoji 拼字、低分辨率识别率低
- **覆盖率**：约 80% 的图片文字场景

### 方案 B：多模态模型（CLIP-based）
```
图片 + 文本 → CLIP 联合编码 → 安全分类头 → 判定
```
- **优点**：端到端理解图文语义组合，不依赖 OCR
- **缺点**：计算成本高，推理慢
- **覆盖率**：能处理 OCR 失败的场景

### 工业界级联方案
```
图片 → OCR 提取文字
  ├── 提取成功 → 文本分类器判定（便宜、快）
  └── 提取失败/不确定 → 多模态模型判定（贵、准）
```

本项目实现两种方案并对比效果，最终推荐级联策略。

---
## 概念 6：Precision-Recall 困境

在内容安全领域，**漏放有害内容的代价远大于误删正常内容**：

| 场景 | 漏放（False Negative） | 误删（False Positive） |
|------|---------------------|---------------------|
| 代价 | 平台声誉损失、法律风险、用户受害 | 用户投诉、创作者不满 |
| 可补救性 | 难以补救 | 可通过申诉恢复 |
| 量级影响 | 一条恶性内容可能造成重大影响 | 误删后可人工复核 |

### 工业界的选择

**生产系统选择 High-Recall 配置**：
- Recall > 0.90（漏放率 < 10%）
- 用人工复核补救 False Positive
- 宁可多拦截，也不能漏放

### 本项目的评估指标设计

| 评估集 | 核心指标 | 目标 |
|-------|---------|------|
| WildGuardTest | AUC-ROC | > 0.85 |
| HarmBench | Recall | > 0.90 |
| XSTest | Over-refusal Rate | < 10% |
| MM-SafetyBench | ASR（攻击成功率） | < 20% |

---
## 概念 7：消融实验——证明每个环节都有用

### 什么是消融实验（Ablation Study）？

"去掉一个组件，看效果变了多少。" 如果去掉后效果明显变差，说明这个组件有用。

### 为什么消融实验重要？

技术交流中最常被问到的问题之一：**"你怎么知道每个环节都有用？"**

回答不能只是"我觉得"，需要有定量数据支撑：

```
"去掉对比无害数据后，Over-refusal Rate 从 8% 升到 35%，
 证明对比构造对防止过度拒绝是必要的。"
```

### 本项目的消融实验设计

| 实验名 | 去掉什么 | 预期影响 | 验证什么 |
|--------|---------|---------|----------|
| Full | 无 | baseline | 对照基准 |
| -Safety | WildGuardMix | AUC 大幅下降 | 安全数据是核心 |
| -Contrastive | 对比无害数据 | Over-refusal 升高 | 对比构造必要性 |
| -Augmentation | 合成增强 | 稀有类别 F1 下降 | 增强的价值 |
| -Copyright | 版权数据 | 版权 Recall ≈ 0 | 版权需专门数据 |
| -ToxiGen | ToxiGen | 仇恨言论 F1 略降 | 隐式毒性贡献 |

---
## 项目完整闭环

```
数据构建（Data-Juicer 清洗 + 增强）→ 训练集
  ↓
模型训练（DistilBERT 文本分类器 / CLIP-based 多模态分类器）
  ↓
量化评估（WildGuardTest AUC、HarmBench Recall、XSTest Over-refusal）
  ↓
消融实验（去掉对比数据 → over-refusal 升高；去掉增强数据 → 稀有类别 F1 降低）
  ↓
反馈：发现哪类数据不够 → 回到数据构建优化
```

### 核心产出

1. **安全 SFT 数据集**（`safety_sft_mix.jsonl`）
2. **安全 DPO 偏好对**（`safety_dpo_pairs.jsonl`）
3. **文本安全分类器**（DistilBERT 微调）+ AUC/F1 报告
4. **评估 Dashboard** + 消融实验报告

In [1]:
# 验证项目配置
import sys
sys.path.insert(0, '..')
from src.utils.config_loader import print_config
print_config()

  当前运行模式: SMOKE_TEST
  文本样本数:     2,000
  图文样本数:     500
  合成增强数:     50
  分类器 Epochs:  1
  设备:           mps
  随机种子:       42
